In [ ]:
import numpy as np
import pandas as pd
import random
import matplotlib.pyplot
%matplotlib inline

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [ ]:
df = pd.read_csv('mentalHdataset.csv')
df.isnull().sum()
df = df.dropna()
label = np.array(df['treatment'])
print(label)

[1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 1 0 1 1 0 0 1 1 1 1 1 1 1 1 0 0 1 0 0 1 1 1
 0 1 1 1 0 1 1 1 1 1 1 1 1 0 1 1 0 1 0 0 1 0 1 1 0 1 1 1 1 0 1 0 1 1 1 1 1
 1 1 1 0 1 0 1 1 1 1]


In [ ]:
df = df.drop(['Timestamp', 'Age', 'state','Unnamed: 0','comments','treatment'] ,axis=1)
df['Gender'] = df['Gender'].replace(['Male', 'Female', 'Other'], [0, 1, 2])
df = df.replace(('No','Yes','Maybe','Some of them', 'Don\'t know','Not sure'), (0,1,2,2,2,2))
df['Country'] = df['Country'].replace(['United States', 'Israel'], [0, 1])
df['work_interfere'] = df['work_interfere'].replace(['Rarely', 'Sometimes', 'Often', 'Never'], [0, 1, 2, 3])
df['leave'] = df['leave'].replace(['Very easy', 'Somewhat easy', 'Somewhat difficult','Very difficult'], [0, 1, 2, 3])
df['no_employees'] = df['no_employees'].replace(['26-100', 'More than 1000', '01-May', 'Jun-25', '500-1000','100-500'], [26, 1000, 0, 0, 500, 100])
df['Age_cat'] = df['Age_cat'].replace(['(31, 72]', '(18, 31]'], [31, 18])
df.head()

,Gender,Country,self_employed,family_history,work_interfere,no_employees,remote_work,tech_company,benefits,care_options,...,leave,mental_health_consequence,phys_health_consequence,coworkers,supervisor,mental_health_interview,phys_health_interview,mental_vs_physical,obs_consequence,Age_cat
24,0,0,0,1,0,26,0,1,1,2,...,2,0,0,1,1,0,1,2,0,31
25,0,0,0,1,1,1000,0,0,1,1,...,0,1,0,2,1,0,1,0,0,31
33,0,0,0,1,1,26,1,1,1,1,...,0,2,0,2,2,2,1,2,0,31
45,1,0,0,1,1,26,0,1,1,1,...,1,0,0,2,1,0,0,1,0,31
49,0,0,0,1,0,26,0,1,1,0,...,2,2,0,2,1,0,0,2,0,18


In [ ]:
#splitting the model into training and testing set
X_train, X_test, y_train, y_test = train_test_split(df,
                                                    label, test_size=0.30,
                                                    random_state=101)

In [ ]:
#training a logistics regression model
logmodel = LogisticRegression(C=0.05, class_weight=None, dual=False, fit_intercept=True,
                              intercept_scaling=1, l1_ratio=None, max_iter=100,
                              multi_class='ovr', n_jobs=None, penalty='l2', random_state=0,
                              solver='liblinear', tol=0.0001, verbose=0, warm_start=False)
logmodel.fit(X_train,y_train)
predictions = logmodel.predict(X_test)
print("Accuracy = "+ str(accuracy_score(y_test,predictions)))

Accuracy = 0.6538461538461539


In [ ]:
#defining various steps required for the genetic algorithm
def initilization_of_population(size,n_feat):
    population = []
    for i in range(size):
        chromosome = np.ones(n_feat,dtype=np.bool)
        chromosome[:int(0.3*n_feat)]=False
        np.random.shuffle(chromosome)
        population.append(chromosome)
    return population

def fitness_score(population):
    scores = []
    for chromosome in population:
        logmodel.fit(X_train.iloc[:,chromosome],y_train)
        predictions = logmodel.predict(X_test.iloc[:,chromosome])
        scores.append(accuracy_score(y_test,predictions))
    scores, population = np.array(scores), np.array(population)
    inds = np.argsort(scores)
    return list(scores[inds][::-1]), list(population[inds,:][::-1])

def selection(pop_after_fit,n_parents):
    population_nextgen = []
    for i in range(n_parents):
        population_nextgen.append(pop_after_fit[i])
    return population_nextgen

def crossover(pop_after_sel):
    population_nextgen=pop_after_sel
    for i in range(len(pop_after_sel)):
        child=pop_after_sel[i]
        child[3:7]=pop_after_sel[((i+1)%len(pop_after_sel))][3:7]
        population_nextgen.append(child)
    return population_nextgen

def mutation(pop_after_cross,mutation_rate):
    population_nextgen = []
    for i in range(0,len(pop_after_cross)):
        chromosome = pop_after_cross[i]
        for j in range(len(chromosome)):
            if random.random() < mutation_rate:
                chromosome[j]= not chromosome[j]
        population_nextgen.append(chromosome)
    return population_nextgen

def generations(size,n_feat,n_parents,mutation_rate,n_gen,X_train,
                                   X_test, y_train, y_test):

    best_chromo= []
    best_score= []
    population_nextgen=initilization_of_population(size,n_feat)
    for i in range(n_gen):
        scores, pop_after_fit = fitness_score(population_nextgen)
        print(scores[:2])
        pop_after_sel = selection(pop_after_fit,n_parents)
        pop_after_cross = crossover(pop_after_sel)
        population_nextgen = mutation(pop_after_cross,mutation_rate)
        best_chromo.append(pop_after_fit[0])
        best_score.append(scores[0])
    return best_chromo,best_score

In [ ]:
chromo,score=generations(size=200,n_feat=df.shape[1],n_parents=100,mutation_rate=0.10,
                     n_gen=3,X_train=X_train,X_test=X_test,y_train=y_train,y_test=y_test)

logmodel.fit(X_train.iloc[:,chromo[-1]],y_train)
predictions = logmodel.predict(X_test.iloc[:,chromo[-1]])
print("Accuracy score after genetic algorithm is= "+str(accuracy_score(y_test,predictions)))
print("The mask is the following", chromo[-1])


<ipython-input-11-05d8049f28f7>:5: DeprecationWarning: `np.bool` is a deprecated alias for the builtin `bool`. To silence this warning, use `bool` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.bool_` here.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  chromosome = np.ones(n_feat,dtype=np.bool)


[0.6538461538461539, 0.6538461538461539]
[0.6538461538461539, 0.6538461538461539]
[0.6538461538461539, 0.6538461538461539]
Accuracy score after genetic algorithm is= 0.6538461538461539
The mask is the following [ True  True  True False False  True False  True False  True  True False
  True False  True False  True  True False False False False  True]
